# CV Masterclass 01: Convolutional Lineage & Mathematics

Welcome to the first notebook of the Computer Vision Deep Dive Sequence.

We do not jump straight into `torch.nn.Conv2d`. To understand why computer vision models are built the way they are, we must trace the entire lineage of spatial filtering. We will build from Dense Layers, identify their catastrophic failure, invent the 2D Convolution, and evolve it mathematically into Dilated and Depthwise Separable Convolutions.

---

## 🎓 Socratic Deep Check
Ponder this question as we progress:

> *"Standard Convolutions computationally explode when the channel depth gets too large. Why does splitting a single $3\times3$ convolution into a 'Depthwise' spatial filter followed by a 'Pointwise' $1\times1$ filter mathematically reduce the parameters by a factor of 8 or 9, while preserving the exact same accuracy?"*

---

## Table of Contents
1. **Dense Layers (The Spatial Failure):** parameter explosion and loss of topology.
2. **Standard Convolutions (The Spatial Prior):** Local connectivity, and Weight Sharing.
3. **The Output Shape Formula (Derived):** Mathematical striding footprint.
4. **Padding Modes: Valid, Same, Full:** Manual NumPy implementations.
5. **Grouped Convolutions (The Bridge):** Interpolating across channels.
6. **Dilated (Atrous) Convolutions:** Expanding Receptive Fields freely.
7. **Depthwise Separable Convolutions:** The MobileNet optimization.
8. **Deep Socratic Synthesis:** Synthesizing striding tradeoffs.
9. **Core Concepts & Pitfalls Summary.**


## 1. Dense Layers (The Spatial Failure)

Imagine a tiny $224 \times 224$ RGB image. Flattened, it is a 1D vector of length $150,528$.

If we feed this into a single standard Dense layer with just 1,000 neurons, the Weight Matrix shape is `[150528, 1000]`. That is **150.5 Million parameters** for a single pitiful layer.

**The Loss of Topology:** A dense layer treats the pixel at geometric coordinate `[0, 0]` as completely mathematically unrelated to the pixel right next to it at `[0, 1]`. All concept of 2D space is instantly destroyed when you flatten the image. The model cannot naturally detect corners or edges.

### ⚠️ Common Pitfalls
*   **Assuming flattening is harmless:** Practitioners sometimes flatten spatial data too early in custom architectures, permanently destroying structural relationships that later layers could have exploited.


## 2. Standard Convolutions (The Spatial Prior)

A Convolution solves both flaws of the Dense layer through two rules:
1.  **Local Connectivity:** A neuron is only mathematically allowed to look at a tiny local patch (e.g., $3\times3$) of the previous layer. 
2.  **Weight Sharing:** The exact same $3\times3$ weight kernel is physically dragged across the entire image. If the kernel learns to detect a vertical edge at coordinate `[0,0]`, it automatically perfectly detects a vertical edge at `[200, 200]`. This is called **Translation Invariance**.

### The Shapes (1D, 2D, 3D)
*   **1D Conv:** Used for Audio/Time-Series data. The kernel slides in 1 geometric direction (Time).
*   **2D Conv:** Used for Images. The kernel slides in 2 directions (Width, Height). 
    *   *Note: A 2D Conv physically processes the Color Channels (Depth) simultaneously, but it does NOT "slide" across the channels. It sums them up. Hence, it is 2D, not 3D.*
*   **3D Conv:** Used for Videos or Medical CT Scans. The kernel physically slides across Width, Height, AND Time (or physical Depth in an MRI scan).

### ⚠️ Common Pitfalls
*   **2D vs 3D Convolution confusion:** Believing a 2D Conv on RGB images is a 3D conv because the input has 3 dimensions (W, H, C). A 2D Conv is strictly 2D because it slides in 2 spatial dimensions, computing a dot product covering the full depth `C` at every stop!


## 3. The Output Shape Formula (Derived)

When sliding a kernel of size $K$ across an image of width $W$, we step by a stride $S$, potentially adding $P$ padding pixels to each side.

**The Derivation:**
1. The padded image width is $W + 2P$.
2. The kernel spans $K$ pixels, so there are an initial $(W + 2P - K)$ pixels *remaining* for the kernel's left edge to slide into.
3. If we jump by $S$ pixels at a time, the number of steps we can take is $(W + 2P - K) / S$.
4. We add $+1$ because we have to count the *first* initial placement at index 0.

**The Golden Formula:**  
`output_size = floor( (W - K + 2P) / S ) + 1`

Let's walk through concrete examples:
1. **Classic MNIST Conv:** $W=28$, $K=3$, $P=0$, $S=1$
   - `floor((28 - 3 + 0) / 1) + 1 = 26` -> Output is $26 \times 26$.
2. **Same-Padding (No Shrinkage):** $W=224$, $K=3$, $P=1$, $S=1$
   - `floor((224 - 3 + 2) / 1) + 1 = 224` -> Perfectly preserves $224 \times 224$.
3. **Strided Conv as Pooling Replacement:** $W=224$, $K=3$, $P=0$, $S=2$
   - `floor((224 - 3 + 0) / 2) + 1 = 111` -> Halves the spatial size (approx). Often used with $K=3, P=1, S=2$ to get exactly $112$, mimicking `MaxPool2d`.

Let's mathematically prove that a strided convolution with $S=2$ shrinks an activation map equivalently to a `MaxPool2d` while keeping learnable parameters.


In [ ]:
import torch
import torch.nn as nn

# Strided Conv (reduces spatial dims directly directly holding parameters)
strided_conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1, stride=2)

# Standard Conv + MaxPool2d (the old way to reduce spatial dims)
conv_and_pool = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1, stride=1),
    nn.MaxPool2d(kernel_size=2, stride=2)
)

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_image = torch.randn(1, 3, 224, 224) # Batch, Channels, Height, Width

out_strided = strided_conv(dummy_image)
out_pooled = conv_and_pool(dummy_image)

print(f"Original Image: {dummy_image.shape}")
print(f"Strided Conv Output:  {out_strided.shape}")
print(f"Conv+MaxPool Output:  {out_pooled.shape}")

assert out_strided.shape == out_pooled.shape, "Shapes do not match!"


### ⚠️ Common Pitfalls
*   **Off-by-one errors with Even Kernels:** Using an even kernel size (like $2\times2$ or $4\times4$) makes symmetric padding mathematically impossible without shifting the image center. Stick to odd kernels ($3, 5, 7$) with padding $P = (K-1)/2$.


## 4. Padding Modes: Valid, Same, Full

*   **Valid:** No padding at all ($P=0$). The output physically shrinks.
*   **Same:** Padding is added so that if $S=1$, the output width exactly matches the input width.
*   **Full:** Maximum padding ($P=K-1$) so the kernel visits every possible interaction, even just 1 pixel on the edge.

To truly understand how a convolution operates, we will build a `manual_2d_conv` function entirely from scratch using raw NumPy and nested loops. We will physically slide the kernel.


In [ ]:
import numpy as np
import torch.nn.functional as F

def manual_2d_conv(image, kernel, padding=0, stride=1):
    """
    A slow, manual NumPy implementation of a single-channel 2D Convolution.
    image: (H, W)
    kernel: (K, K)
    """
    H, W = image.shape
    K = kernel.shape[0]
    
    # Apply padding
    padded_image = np.pad(image, pad_width=padding, mode='constant', constant_values=0)
    
    out_H = int(np.floor((H + 2 * padding - K) / stride)) + 1
    out_W = int(np.floor((W + 2 * padding - K) / stride)) + 1
    
    output = np.zeros((out_H, out_W))
    
    # Physically slide the window!
    for y in range(out_H):
        for x in range(out_W):
            # Find the starting indices on the padded image
            start_y = y * stride
            start_x = x * stride
            
            # Extract the receptive patch
            patch = padded_image[start_y : start_y+K, start_x : start_x+K]
            
            # Element-wise multiply and sum (Dot Product)
            output[y, x] = np.sum(patch * kernel)
            
    return output

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
# Create random 5x5 image and 3x3 kernel
image_np = np.random.rand(5, 5).astype(np.float32)
kernel_np = np.random.rand(3, 3).astype(np.float32)

# Manual Run
out_manual = manual_2d_conv(image_np, kernel_np, padding=1, stride=1)
print(f"Manual Output Shape: {out_manual.shape}")

# PyTorch Run (needs batch and channel dims)
image_th = torch.from_numpy(image_np).unsqueeze(0).unsqueeze(0) # (1, 1, 5, 5)
kernel_th = torch.from_numpy(kernel_np).unsqueeze(0).unsqueeze(0) # (1, 1, 3, 3)

out_torch = F.conv2d(image_th, kernel_th, padding=1, stride=1)
out_torch_np = out_torch.squeeze().numpy()

print(f"PyTorch Output Shape: {out_torch_np.shape}")
print(f"Max Difference: {np.max(np.abs(out_manual - out_torch_np)):.6f}")

assert np.allclose(out_manual, out_torch_np, atol=1e-4), "Manual convolution failed to match PyTorch!"


### ⚠️ Common Pitfalls
*   **"Same" Padding with Strides:** Setting PyTorch padding mode to `'same'` will crash if you use a stride $>1$. "Same" mathematically implies preserving output dimensions, which is impossible if you are skipping pixels.


## 5. Grouped Convolutions (The Bridge)

A standard 2D convolution combines spatial filtering and channel cross-mixing into a single operation. If we have $C_{in}$ channels, the kernel physically evaluates all $C_{in}$ channels simultaneously.

**Grouped Convolutions** act as a mathematical bridge or interpolation between a standard convolution and a depthwise convolution:
- `groups = 1`: Standard convolution. 1 massive group sees all channels.
- `groups = 2`: Input channels are split in half. Two kernels operate independently, one per half.
- `groups = C_in`: **Depthwise Convolution**. Every individual channel gets its own independent spatial kernel. They NEVER mix!

**The Parameter Count Formula:**
- Standard parameters: $C_{in} \times C_{out} \times K \times K$
- Grouped parameters: $\frac{C_{in}}{g} \times C_{out} \times K \times K$


In [ ]:
# Compare Grouped Conv parameter counts
c_in = 512
c_out = 512
kernel_size = 3

conv_standard = nn.Conv2d(c_in, c_out, kernel_size, groups=1, bias=False)
conv_grouped = nn.Conv2d(c_in, c_out, kernel_size, groups=4, bias=False)
conv_depthwise = nn.Conv2d(c_in, c_out, kernel_size, groups=c_in, bias=False)

def get_params(layer):
    return sum(p.numel() for p in layer.parameters())

print(f"Standard (groups=1):    {get_params(conv_standard):,} params")
print(f"Grouped (groups=4):     {get_params(conv_grouped):,} params")
print(f"Depthwise (groups=512): {get_params(conv_depthwise):,} params")

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy_512 = torch.randn(1, 512, 16, 16)
out_dw = conv_depthwise(dummy_512)
print(f"\nDepthwise Output Shape: {out_dw.shape}")

# Verify math exactly matches formula
expected_dw_math = int((c_in / c_in) * c_out * 3 * 3)
assert get_params(conv_depthwise) == expected_dw_math, "Formula mismatch!"


### ⚠️ Common Pitfalls
*   **Confusing `groups` with `out_channels`:** `groups=4` does not output 4 channels. It divides the *input* channels into 4 isolated lanes. The output channels are also divided equally amongst those lanes!


## 6. Dilated (Atrous) Convolutions

If you want to increase an imaging network's "Receptive Field" (how much of the original image a deep neuron can 'see'), you typically use a Max-Pooling layer. This compresses a $224\times224$ map down to $112\times112$. 

**The Problem:** In Semantic Segmentation (where you need a perfectly crisp $224\times224$ mask output), throwing away $75\%$ of your spatial pixels during max pooling is catastrophic.

**The Atrous Fix:** Invented for DeepLab, a Dilated Convolution physically injects empty spaces "holes" into the $3\times3$ kernel. The 9 weights spread out and cover a $5\times5$ spatial area by simply skipping the pixels in between. 
**Result:** The Receptive Field physically triples, but the spatial resolution of the feature map never shrinks. No max-pooling is required!


In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: Dilated Convolution vs Standard Conv
# -----------------------------------------------------

def demonstrate_dilation():
    # Standard 3x3 Kernel on a 1D slice
    standard_mask = np.array([1, 1, 1])
    print(f"Standard 1D Kernel touches adjacent pixels: {standard_mask}")
    print(f"Total Pixels Spanned: {len(standard_mask)}")
    
    # Dilation Rate = 2 (Injects 1 hole between every weight)
    dilated_mask = np.array([1, 0, 1, 0, 1])
    print(f"\nDilated 1D Kernel touches spread pixels:    {dilated_mask}")
    print(f"Total Pixels Spanned: {len(dilated_mask)}")
    print("\nNotice that BOTH kernels contain exactly 3 trainable weights (parameters).")
    print("But the Dilated kernel spans nearly double the physical field of view.")

demonstrate_dilation()

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
# In PyTorch, observe that dilation impacts output shape equivalent to a larger kernel
dilated_conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, dilation=2, bias=False)
dummy = torch.randn(1, 1, 7, 7)
out = dilated_conv(dummy)
print(f"\nInput: {dummy.shape}")
print(f"Output with K=3, Dilation=2: {out.shape} -> (Equivalent to K=5 output shape without padding)")


### ⚠️ Common Pitfalls
*   **Gridding Artifacts:** Using heavy dilations consecutively without any dense mixing layers can result in "gridding" artifacts where sparse isolated sub-grids never communicate with neighboring pixels.


## 7. Depthwise Separable Convolutions

A standard `Conv2d` layer taking an RGB image (3 channels) and outputting 64 feature maps uses $64$ separate $3\times3\times3$ kernels.

Deep in a ResNet, a layer might take 512 channels and output 512 channels. 
**Standard Params:** $3 \times 3 \times 512 \times 512 = 2.3 \text{ Million parameters}$.

Google's **MobileNet** architecture revolutionized Edge AI by factoring that monolithic operation into two mutually exclusive steps:

### Step A: Depthwise Convolution
Instead of looking at all 512 input channels at once, we apply one single strictly isolated 2D $3\times3$ filter to Channel 1. Another isolated $3\times3$ filter to Channel 2. And so on.
**Cost:** $3 \times 3 \times 512 = 4,608 \text{ parameters}$. 
**Output:** 512 spatially filtered channels. But they have *never communicated with each other*. There is no cross-color intelligence.

### Step B: Pointwise Convolution
To fix the lack of cross-group intelligence, we apply a massive $1\times1$ Convolution across all 512 channels. It mathematically compresses and combines the features across the depth dimension, outputting the final 512 maps.
**Cost:** $1 \times 1 \times 512 \times 512 = 262,144 \text{ parameters}$.

### Factorization Math
Total parameters of Depthwise Separable: $4,608 + 262,144 = 266,752$.
Efficiency Gain: $2.35M / 266k = \approx 8.8\times$.


In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # Step A: Spatial feature extraction per-channel
        self.depthwise = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )
        
        # Step B: Channel mixing
        self.pointwise = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

def compare_efficiency(c_in, c_out):
    std_conv = nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False)
    ds_conv = DepthwiseSeparableConv(c_in, c_out)
    
    std_params = sum(p.numel() for p in std_conv.parameters())
    
    # Ignoring BN params to focus on raw mult-adds weight footprint for the ratio
    # We will compute strictly standard vs DW+PW conv for fairness and math clarity
    ds_conv_only_params = (c_in * 3 * 3) + (c_in * c_out * 1 * 1)
    
    ratio = std_params / ds_conv_only_params
    print(f"C={c_in:<4}: Std Params = {std_params:<10,} | DS Params = {ds_conv_only_params:<10,} | Efficiency Ratio: {ratio:.1f}x")

print("Parameter Footprint Face-off:")
compare_efficiency(64, 64)
compare_efficiency(256, 256)
compare_efficiency(512, 512)

# -----------------------------------------------------
# TEST IT
# -----------------------------------------------------
dummy = torch.randn(2, 512, 14, 14)
ds_module = DepthwiseSeparableConv(512, 512)
out_ds = ds_module(dummy)
print(f"\nForward pass via DS Module Output: {out_ds.shape}")


### ⚠️ Common Pitfalls
*   **Forgetting BN and ReLU between steps:** Originally researchers skipped the activation between depthwise and pointwise. MobileNet proved that injecting `BatchNorm -> ReLU` between them significantly stabilizes edge training. 


## 8. 🎓 Deep Socratic Synthesis

**Q:** *A `Conv2d` with `kernel_size=3, padding=1, stride=1` preserves spatial dimensions. But a `Conv2d` with `kernel_size=3, padding=0, stride=2` AND a `Conv2d` with `kernel_size=1, padding=0, stride=2` both halve spatial dimensions. What is the computational cost difference between these two approaches, and why do architectures like MobileNetV2 use the `stride=1` with `1x1` projection rather than a strided `3x3`?*

**Ponder deeply:** 
- A $3\times3$ strided convolution drops spatial dimension while executing $9 \times C_{in} \times C_{out}$ multiplications per stop.
- A $1\times1$ strided convolution drops spatial dimension entirely using only $1 \times C_{in} \times C_{out}$ multiplications per stop (an immediate $9\times$ savings), delegating the actual dense spatial understanding to a fast depthwise step!


## 9. Core Concepts & Pitfalls Summary

Before proceeding to Advanced Architectures in Notebook 02, verify you do not harbor any of these dangerous misconceptions:

### ❌ PITFALL 1: Expecting `Conv2d` to loop over channels.
Forgetting that a 2D Conv processes all input channels simultaneously. It is NOT a per-channel operation by default. When you provide a `Conv2d` with an image tensor shaped `(3, 224, 224)`, it computes a dense dot product encompassing all 3 color planes inside its spatial area, outputting a scalar. To enforce channel independence, you MUST formulate a depthwise convolution via `groups=in_channels`.

### ❌ PITFALL 2: Confusing `groups` with Output Channel splitting.
Believing that setting `groups=4` implies the network outputs 4 branches. It actually slices your *input* channels into 4 isolated lanes. A 512-channel input with `groups=4` yields 4 lanes of 128 channels. These mathematically never cross-communicate unless combined later.

### ❌ PITFALL 3: Designing blindly with Transposed Convolutions.
When attempting to upsample tensors, using a standard fractional stride (Transposed Convolution or Deconvolution) can create dangerous Checkerboard Artifacts. Due to overlapping unevenly, raw pixels generate intense frequency gradients. **Preview for NB04:** Always favor a deterministic `nn.Upsample` combined with a `stride=1` `Conv2d` for premium generative models.
